# 04 — Ablation Study

Full ablation study comparing:
1. NLI-only vs Embedding-only vs LLM-judge-only vs Ensemble
2. Weight sensitivity analysis
3. Threshold optimization
4. Results table generation for the paper

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.utils.data_loader import load_halueval
from src.detection.nli_detector import NLIDetector
from src.detection.embedding_detector import EmbeddingDetector
from src.detection.llm_judge import LLMJudge
from src.detection.ensemble import EnsembleDetector
from src.evaluation.metrics import (
    hallucination_detection_auroc,
    hallucination_detection_f1,
)

In [ ]:
# Load evaluation data
data = load_halueval(split='test')[:300]
y_true = [s['label'] for s in data]
print(f'Samples: {len(data)}, Faithful: {y_true.count(0)}, Hallucinated: {y_true.count(1)}')

## 1. Individual Detector Comparison

In [ ]:
# NLI scoring
nli = NLIDetector()
nli_scores = [nli.score(s['response'], s['context']) for s in tqdm(data, desc='NLI')]

# Embedding scoring
emb = EmbeddingDetector()
emb_scores = [emb.score(s['response'], s['context']) for s in tqdm(data, desc='Embedding')]

In [ ]:
# Ablation results table
ablation_results = []

for name, scores in [('NLI-only', nli_scores), ('Embedding-only', emb_scores)]:
    auroc = hallucination_detection_auroc(y_true, scores)
    f1_result = hallucination_detection_f1(y_true, scores)
    ablation_results.append({
        'Method': name,
        'AUROC': auroc,
        'F1': f1_result['f1'],
        'Precision': f1_result['precision'],
        'Recall': f1_result['recall'],
    })

df_ablation = pd.DataFrame(ablation_results)
print(df_ablation.to_string(index=False))

## 2. Weight Sensitivity Analysis

In [ ]:
# Test different weight configurations
weight_configs = [
    {'nli': 0.33, 'embedding': 0.33, 'llm_judge': 0.34},
    {'nli': 0.5, 'embedding': 0.2, 'llm_judge': 0.3},
    {'nli': 0.6, 'embedding': 0.1, 'llm_judge': 0.3},
    {'nli': 0.7, 'embedding': 0.15, 'llm_judge': 0.15},
    {'nli': 0.4, 'embedding': 0.3, 'llm_judge': 0.3},
]

# Simulate ensemble scores using pre-computed individual scores
for config in weight_configs:
    # Note: This is a simplified simulation using only NLI and Embedding
    # Full ensemble would include LLM Judge scores
    ensemble_scores = [
        config['nli'] * nli + config['embedding'] * emb + config['llm_judge'] * 0.5
        for nli, emb in zip(nli_scores, emb_scores)
    ]
    auroc = hallucination_detection_auroc(y_true, ensemble_scores)
    print(f'Weights {config} → AUROC: {auroc:.4f}')

## 3. Threshold Optimization

In [ ]:
# Test different thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []

for t in thresholds:
    f1_result = hallucination_detection_f1(y_true, nli_scores, threshold=t)
    threshold_results.append({
        'Threshold': round(t, 2),
        'F1': f1_result['f1'],
        'Precision': f1_result['precision'],
        'Recall': f1_result['recall'],
    })

df_threshold = pd.DataFrame(threshold_results)
best_idx = df_threshold['F1'].idxmax()
print(f'Best threshold: {df_threshold.loc[best_idx, "Threshold"]} (F1={df_threshold.loc[best_idx, "F1"]:.4f})')
print(df_threshold.to_string(index=False))

## 4. Final Paper Results Table

In [ ]:
# Generate final results table for the paper
print('\n=== Paper Results Table ===')
print(df_ablation.to_latex(index=False, float_format='%.4f'))